# Real Simulation of a Single Segment

This notebook uses the real project code and a valid Lookup Table (LUT) to simulate the detector response for a single particle segment from `event122.h5`.

In [2]:
import sys

# Add project root to path
project_root = '/sdf/group/neutrino/pgranger/larnd-sim-jax/src'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

%cd /sdf/group/neutrino/pgranger/larnd-sim-jax


import jax.numpy as jnp
import jax
import matplotlib.pyplot as plt
import numpy as np
import h5py

from larndsim.consts_jax import build_params_class, load_detector_properties, load_lut
from larndsim.sim_jax import simulate_wfs, prepare_tracks



# 1. Load Parameters and LUT
detector_props = "src/larndsim/detector_properties/module0.yaml"
pixel_layouts = "src/larndsim/pixel_layouts/multi_tile_layout-2.4.16_v4.yaml"
lut_file = "src/larndsim/detector_properties/response_44_v2a_full_tick.npz"

print("Loading parameters...")
params_cls = build_params_class([])
params = load_detector_properties(params_cls, detector_props, pixel_layouts)
params = params.replace(
    electron_sampling_resolution=0.01
)

# Update some critical simulation parameters
params = params.replace(
    electron_sampling_resolution=0.01, # cm
    number_pix_neighbors=1,           # 3x3 grid
    signal_length=60,                 # ticks
    t_sampling=0.1                    # us
)

print("Loading LUT (this might take a moment)...")
response_template, params = load_lut(lut_file, params)
print(f"Response template shape: {response_template.shape}")

/sdf/home/p/pgranger/.local/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/sdf/group/neutrino/pgranger/larnd-sim-jax
Loading parameters...


INFO:larndsim.consts_jax:Loading response from npz file


Loading LUT (this might take a moment)...


Could not load symbol cuFuncGetName. Error: /.singularity.d/libs/libcuda.so.1: undefined symbol: cuFuncGetName


Response template shape: (100, 45, 45, 1950)


In [3]:
# 2. Prepare the Track from event122.h5
input_file = "event122.h5"
print(f"Loading track from {input_file}...")
tracks, fields, original_tracks = prepare_tracks(params, input_file, invert_xz=True)

print(f"Number of mini-segments after chopping: {tracks.shape[0]}")
print(f"Fields: {fields}")

Loading track from event122.h5...
Number of mini-segments after chopping: 5526
Fields: ('eventID', 'z_end', 'trackID', 'tran_diff', 'z_start', 'x_end', 'y_end', 'n_electrons', 'pdgId', 'x_start', 'y_start', 't_start', 'dx', 'range', 'long_diff', 'pixel_plane', 't_end', 'dEdx', 'dE', 't', 'y', 'x', 'z')


In [4]:
# 3. Run the Simulation
print("Running simulation...")
wfs, unique_pixels = simulate_wfs(params, response_template, tracks, fields)

print(f"Simulation complete.")
print(f"Number of active pixels: {unique_pixels.shape[0]}")
print(f"Waveform shape: {wfs.shape}")

Running simulation...


ValueError: tuple.index(x): x not in tuple

In [ ]:
# 4. Visualize the Result
# Find the pixel with the maximum signal
max_val_idx = jnp.argmax(jnp.max(wfs, axis=1))
max_wf = wfs[max_val_idx]
pix_id = unique_pixels[max_val_idx]

plt.figure(figsize=(12, 6))

# Plot the strongest pixel
plt.plot(max_wf, label=f"Strongest Pixel (ID: {pix_id})", lw=2, color='blue')

# Plot some neighbors if they exist
for i in range(min(5, wfs.shape[0])):
    if i != max_val_idx:
        plt.plot(wfs[i], alpha=0.5, ls='--', label=f"Neighbor (ID: {unique_pixels[i]})")

plt.title(f"Real Simulated Waveforms for segment in {input_file}")
plt.xlabel("Time Ticks (0.1 us)")
plt.ylabel("Induced Current")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Also show a 2D plot of the peak current per pixel
plt.figure(figsize=(6, 5))
peak_currents = jnp.max(wfs, axis=1)
plt.scatter(range(len(peak_currents)), peak_currents, c=peak_currents, cmap='viridis')
plt.colorbar(label="Peak Current")
plt.title("Peak Current per Active Pixel")
plt.xlabel("Pixel Index")
plt.ylabel("Current")
plt.show()